# <center> Data Science in Healthcare : Breast Cancer Detection <center/>
<center> <b>DLBDSME01<b/> - Model Engineering <center/>
<center> IU International University of Applied Sciences <center/>

## Objective :
Greetings, in this project, our goal is to help our local oncologists address the interpretability challenges of these systems by developing a transparent ML model capable of classifying tumors as Malignant or Benign while ensuring explainability in decision-making.

In this notebook, we're proposing an integration strategy to support daily diagnostics using our model.

## List of contents :
1. Suggested Clinical Workflow
2. API Design
3. Containerization
4. Dashboard
5. Summary

Start with importing the required libraries :

In [1]:
# Importing the required libraries
import warnings


# Avoid unnecessary warnings
warnings.filterwarnings('ignore')

## 1. Suggested Clinical Workflow

* **Data Acquisition & Processing:** A clinical sample is obtained, and the corresponding features (mean, standard deviation, and worst values) are calculated.
* **Dashboard Integration:** The lab makes the processed data available on the oncologist's interface.
* **Automated Prediction:** The oncologist simply clicks the **Predict** button. In the background, the data is securely transmitted via a Flask API to the machine learning model, which calculates a predictive score and assigns a label of either **Benign** or **Malignant**.
* **Results Delivery:** The model's prediction is sent directly back to the oncologist’s dashboard.
* **Final Clinical Assessment:** The oncologist retains ultimate accountability for the diagnosis. They review the prediction, compare the actual sample values against the 99% Confidence Interval (CI) established in the previous phase, and make a final decision to either confirm the diagnosis or request further examinations.

## 2. API Design

The model is deployed as a lightweight Flask service exposing key RESTful endpoints to process incoming cytological measurements, record diagnostic predictions, and log expert clinical reviews.

<img src="../assets/figures/Fig. 10 - API Design.png" alt="API Design" width="2000">

### API Endpoints

* **`GET /`**
  * **Description:** Health check verifying the API service state.
  * **Response:** Returns API version status.

* **`GET /patients`**
  * **Description:** Returns all patient diagnostic records ordered by newest first. Used to populate the dashboard diagnostic queue.

* **`POST /predict`**
  * **Description:** Receives 30 numerical cytological features, runs model prediction, saves record to SQLite db, and returns diagnostic results (Malignant/Benign) with confidence score.

* **`POST /confirm/<record_id>`**
  * **Description:** Records oncologist's confirmation of the model's prediction in the database.

* **`POST /reject/<record_id>`**
  * **Description:** Flags a prediction as rejected and saves mandatory clinical feedback notes.

## 3. Containerization

To ensure reliable, reproducible, and cross-platform model deployment, the Flask service is packaged inside a Docker container. This isolation encapsulates all dependencies, compiler libraries, and pre-trained model binaries.

### Docker Setup

The deployment environment uses a multi-stage execution setup defined in `DockerFile`:
1. **Base Image:** `python:3.10-slim` to minimize container overhead and footprint.
2. **Dependencies:** Installs compiling requirements (`build-essential`, `gfortran`, and linear algebra libraries) to optimize scikit-learn/numpy runtime performance.
3. **Model Packaging:** Copies model binaries (`models/main_model_v1.joblib`), helper modules, data sets, and exposes port `5000`.

### Build and Run Commands

```bash
# Build the container image
docker build -t breast-cancer-detection -f DockerFile .

# Run the container mapping internal port 5000 to external host port 5000
docker run -p 5000:5000 breast-cancer-detection
```

## 4. Dashboard

The clinical frontend is a React-based application designed to bridge AI predictions with the physician's workflow. It features clear information hierarchy, high-contrast layouts, and side-by-side comparative diagnostics.

### Diagnostic Patients List

The left navigation pane allows the oncologist to search, filter, and review the status of diagnostic cases. Borderline scores are flagged to highlight cases requiring close observation.

<img src="../assets/figures/Fig. 11 - Patients List.png" alt="Patients List Dashboard" width="2000">

### Case Details Workspace

The primary diagnostic workspace provides a detailed view of the selected patient. It includes cytological measurements plotted against benign/atypical reference lines, genomic profiles, CA 15-3 antigen timeline, and the prop-driven `DiagnosisResultCard` showing prediction classification and a half-circle confidence gauge.

<img src="../assets/figures/Fig. 12 - Patients Diagnosis Details.png" alt="Patients Diagnosis Details" width="2000">

## Summary & Clinical Synthesis

This project serves as a compelling demonstration of data science’s transformative potential in healthcare. By optimizing the diagnostic workload of interpreting cytological profiles and mammograms, and providing clinicians with real-time, explainable insights through an intuitive, interactive dashboard, these tools empower healthcare professionals to focus on patient-centric care and therapeutic innovation. Implementing transparent, AI-driven solutions of this caliber directly reinforces clinical trust and establishes a robust foundation for next-generation medical systems.

While this was not our first experience engineering AI solutions, incorporating model interpretability represented a fundamental paradigm shift in our development methodology. Historically, predictive modeling often favored optimizing performance metrics at the cost of clarity, treating predictions as a "black box" to be accepted at face value. Integrating explainability has rewritten this workflow; it not only uncovers the underlying "why" behind a prediction but also maps out clear, statistically validated decision boundaries.

Furthermore, collaborating closely with oncologists as domain-expert stakeholders provided invaluable perspectives that forced a critical re-examination of statistical assumptions. Grounding algorithmic features in real-world clinical contexts bridged the gap between pure data science and bedside utility. Ultimately, the profound insights gained from this collaborative study validated every hour of effort, underscoring that medical AI achieves its true potential only when it functions as an interpretable partner to human expertise.

## Author
<a href="https://www.linkedin.com/in/ab0858s/">Abdelali BARIR</a> is a former veteran in the Moroccan's Royal Armed Forces, and a self-taught python programmer. Currently enrolled in B.Sc. Data Science in __IU International University of Applied Sciences__.

## Change Log

| Date         | Version   | Changed By       | Change Description        |
|--------------|-----------|------------------|---------------------------|
| 2026-06-30   | 1.1       | Antigravity AI   | Completed model deployment details, Docker containerization, and dashboard figure references |
| 2026-05-10   | 1.0       | Abdelali Barir   | Started Project           |